__Jose Ramon William T. Munda III, 224401__                 

__CSCI 71 - C__

__Multibalanced Bracket Project__



### Write Up:

Formally, a PDA is defined by a 7-tuple $ M = (Q, \Sigma, \Gamma, \delta, q_0, Z_0, F)$. In my PDA $M'$, these 7 parameters are defined as follows:
- $Q = \{q_0, q_1, q_2\}$; 
- $\Sigma = \Big\{x, ], ), \}, >, [, (, \{, <, ! \Big\}$ 
- $\Gamma = \Big\{[, (, \{, <, !, Z \Big\}$ 
- $\delta: Q \times (\Sigma + \epsilon) \times \Gamma$ 
- $q_0$: `q0` (initial state)
- $Z_0$: "Z" in the initial stack
- $F = q_2$

My implementation of a PDA primarily uses the states and transitions as **functions**. It works by taking an input string *s* and slicing its characters. I take the first character of the string, pass it through a state in $Q$ then pass the remaining string to its next corresponding state according to the PDA. Note that this implementation is also **recursive** so I only need to call the initial state, then it will go through the rest of the PDA. 

The program follows the PDA seen below:

![My Image](Munda_PDA.png)

## Global Variables

These variables are used althroughout the entire code.

`OPEN_BRACKETS`, `CLOSED_BRACKETS` - refers to the open/closed brackets


`STACK_ALPHABET` $(\Gamma)$ - refers to the accepted strings in the stack 


`INPUT_ALPHABET` $(\Sigma)$ - refers to the accepted strings that the PDA takes the input of 


In [ ]:
OPEN_BRACKETS = ["[", "(", "{", "<"]
CLOSED_BRACKETS = ["]", ")", "}", ">"]

STACK_ALPHABET = ["[", "(", "{", "<", "!", "Z"]
INPUT_ALPHABET = ["x", "]", ")", "}", ">", "[", "(", "{", "<", "!"]

## Helper Functions

`id` - prints the instantaneous description of the PDA according to the current state it is in.

In [ ]:
def id(s, curr_state, curr_stack):
    output_stack = ''.join(curr_stack)[::-1]
    print(f"ID: ({curr_state}, {s}, {output_stack})")

## Required Functions

Functions that are required as per the specifications of the project.

### `is_balanced`

This is where the PDA is actually simulated. As mentioned, $Q = \{q_0, q_1, q_2\}$ is expressed as functions.

`q0` - Initial state. It checks if the first character is an exclamation point (!), then moves directly to `q1`.

`q1` - Checks if the character is in the input alphabet $(\Sigma)$. If it is an open bracket, it will put it in the stack (*push*). If it is a closed bracket, it first checks if the top of the stack is the correct corresponding character then it will remove it (*pop*). Once it sees an exclamation point (!), it will move to `q2`.

`q2` - Final state. Checks if all the characters in the string is exhausted. If it is, return `True`.

In [ ]:
def is_balanced(s):
    global original_s 
    original_s = s
    
    balanced_stack = ["Z"]

    def q0(s, curr_stack, position):

        id(s, "q0", curr_stack)

        char = s[0]

        if char == "!":
            curr_stack.append(char)
            position += 1
            return q1(s[1::], curr_stack, position)
        elif char != "!":
            print(f"Invalid string. Failed at position {position}.")
            print(f"Remaining unprocessed input string: {s[0::]}\n")
            return False
        
    def q1(s, curr_stack, position):
        # checks if s is empty (also equivalent to checking if top of the stack is Z0)
        if s == "": 
            s += "E"

        char = s[0]
        remaining_string = s[1::]

        id(s, "q1", curr_stack)

        if char == "E":
            print("Invalid state. q1 is not a final state.\n")
            return False
        
        elif char not in INPUT_ALPHABET:
            print(f"Invalid string. Failed at position {position}.")
            print(f"Remaining unprocessed input string: {s[0::]}\n")
            return False

        elif char == "x":
            position += 1
            return q1(remaining_string, curr_stack, position)
        elif char in OPEN_BRACKETS:
            curr_stack.append(char)
            position += 1
            return q1(remaining_string, curr_stack, position)
        elif char == "!" and curr_stack[-1] == "!":
            curr_stack.pop()
            position += 1
            return q2(remaining_string, curr_stack, position)
        elif char in CLOSED_BRACKETS:
            top = curr_stack[-1]
            if char == "]" and top == "[":
                curr_stack.pop()
                position += 1
                return q1(remaining_string, curr_stack, position)
            elif char == ")" and top == "(":
                curr_stack.pop()
                position += 1
                return q1(remaining_string, curr_stack, position)
            elif char == "}" and top == "{":
                curr_stack.pop()
                position += 1
                return q1(remaining_string, curr_stack, position)
            elif char == ">" and top == "<":
                curr_stack.pop()
                position += 1
                return q1(remaining_string, curr_stack, position)
            else:
                print(f"Invalid string. Failed at position {position}.")
                print(f"Remaining unprocessed input string: {s[0::]}\n")
                return False
        
        return False
        
    def q2(s, curr_stack, position):
        # checks if s is empty (also equivalent to checking if top of the stack is Z0)
        if s == "":
            s += "E"

        id(s, "q2", curr_stack)

        if s != "E":
            print(f"Invalid string. Failed at position {position}.")
            print(f"Remaining unprocessed input string: {s[0::]}\n")
            return False
        else:
            print("q2 is a final state.")
            print(f"{original_s} is valid and has balanced brackets. \n")
            return True
        
    # As mentioned in the write up, we only need to call the initial state. The rest will follow.
    return q0(s, balanced_stack, 1)

### `evaluate`

Given that a string is balanced, this implementation looks for the very first instance of a closed bracket, and evaluates the operation of that corresponding bracket.


In [ ]:
def evaluate(s):
    evaluate_stack = []

    for char in s:

        if char in OPEN_BRACKETS:
            evaluate_stack.append(char)
        elif char == "!":
            continue
        elif char == "x":
            evaluate_stack.append("x")
        else:
            temp = ""

            while evaluate_stack and evaluate_stack[-1] not in OPEN_BRACKETS:
                temp = evaluate_stack.pop() + temp  

            if evaluate_stack:
                evaluate_stack.pop() 

            if char == ">":
                evaluate_stack.append(temp * 2)
            elif char == "}":
                evaluate_stack.append(temp + "x")
            elif char == "]":
                evaluate_stack.append("") 
            elif char == ")":
                evaluate_stack.append(temp[1:] if temp else "")

    result = "".join(evaluate_stack)
    l = len(result)
    return l

### `main1`

This function simply takes in inputs from a text file `input.txt` then processes each input line by line with `is_balanced`.

In [ ]:
def main1():
    with open("input.txt", "r") as file:
        for line in file:
            s = line.strip()
            print(f"Processing {s}")
            is_balanced(s)

### `main2`

Like `main1`, it takes in inputs from `input.txt`, but this time, if `is_balanced` returns `True`, we also run `evaluate`.

In [ ]:
import sys
import io

def main2():
    with open("input.txt", "r") as file:
        for line in file:
            s = line.strip()

            # Temporarily suppress prints from is_balanced
            old_stdout = sys.stdout
            sys.stdout = io.StringIO()

            valid = is_balanced(s)

            # Restore normal printing
            sys.stdout = old_stdout

            if valid:
                result = evaluate(s)
                print(f"{s} - Resulting number of x's: {result}")
            else:
                print(f"{s} - Invalid string.")

In [ ]:
# For Running
main1()
main2()